## Data Cleaning & Feature Engineering

Raw data is never ready for machine learning. This step cleans inconsistencies, unifies formats, and then engineers features — calculating new columns that describe patterns the raw data doesn't directly contain.

In [33]:
# Import required library
import pandas as pd
from pathlib import Path

# load the cleaned data saved at the end of the Data Extraction notebook
OUT_DIR = Path(r"C:\Users\hp\Basic_healthcare_modelling_project\output")  # same OUT_DIR as extraction notebook

# Load the 3 quarterly disbursement files
q1 = pd.read_csv(OUT_DIR / "q1_disbursements.csv")
q2 = pd.read_csv(OUT_DIR / "q2_disbursements.csv")
q3 = pd.read_csv(OUT_DIR / "q3_disbursements.csv")

# Clean Q1 and Q2
def clean_quarterly(df, quarter_name):
    
    # Drop rows where state is NaN or "NAN" or contains "TOTAL"
    df = df[~df["state"].astype(str).str.upper().isin(["NAN", "TOTAL", "GRAND TOTAL", ""])].copy()
    df = df[~df["state"].astype(str).str.upper().str.contains("TOTAL", na=False)].copy()
    df = df.dropna(subset=["state"]).copy()

    # Rename columns to match Q3
    df = df.rename(columns={
        "nhia_total":  "nhia",
        "grand_total": "total_alloc",
    })

    # Add missing emtc column (Q1/Q2 Excel had none — treat as 0)
    df["emtc"] = 0.0

    # Keep only the 7 standard columns
    df = df[["state", "nphcda", "ncdc", "emtc", "nhia", "total_alloc", "quarter"]].copy()
    df["quarter"] = quarter_name
    return df

q1_clean = clean_quarterly(q1, "Q1")
q2_clean = clean_quarterly(q2, "Q2")
q3_clean = q3[["state", "nphcda", "ncdc", "emtc", "nhia", "total_alloc", "quarter"]].copy()

# Stack into one long-format DataFrame (111 rows: 37 states × 3 quarters)
df = pd.concat([q1_clean, q2_clean, q3_clean], ignore_index=True)

print("Shape:", df.shape)        # should be (111, 7)
print("Quarters:", df["quarter"].value_counts().to_dict())
print("NaN check:\n", df.isnull().sum())
df.head()

# Load facilities separately
facilities = pd.read_csv(OUT_DIR / "facilities.csv")

Shape: (111, 7)
Quarters: {'Q1': 37, 'Q2': 37, 'Q3': 37}
NaN check:
 state          0
nphcda         0
ncdc           0
emtc           0
nhia           0
total_alloc    0
quarter        0
dtype: int64


In [34]:
from scipy.stats import entropy   # for calculating shannon diversity

# Government documents spell state names differently across files.
# This dictionary maps every known variation to one standard spelling
State_Canonical = {
    "ABUJA": "FCT",     # The capital is sometimes called ABUJA
    "F.C.T" : "FCT",    # sometimes abbreviated with dots
    "FEDERAL CAPITAL TERRITORY": "FCT",   # sometimes spelled out in full
    "AKWA-IBOM": "AKWA IBOM",   # hyphen vs space
    "CROSS-RIVER": "CROSS RIVER",
    "NASSARAWA": "NASARAWA"   # common mispelling in government docs
}

def canonical_state(name):
    # Strip whitespace, make uppercase, remove the word "STATE"
    # e.g. "Lagos State" -> "LAGOS STATE" -> "LAGOS"
    s = str(name).strip().upper().replace("STATE", "").strip()
    return State_Canonical.get(s, s)

# Apply this function to the "state" column in all four DataFrames.
# This ensures LAGOS in facilities.csv matches LAGOS in q3.csv.
for frame in [q1, q2, q3, facilities]:
    frame["state"] = frame["state"].apply(canonical_state)

    

**BUILDING THE FEATURE TABLE**

In [35]:
# ── FACILITY HEALTH SUMMARY per state ─────────────────────
# For each state, calculate what % of its PHCs are in each status tier.
# This gives the ML model a single number to represent "how healthy" a
# state's health infrastructure is.
FUNC_WEIGHT = {
    "L2": 1.0,    # fully functional - best outcome
    "L1": 0.75,   # functional level 1
    "PARTIAL": 0.40,   # partially working
    "NON-FUNCTIONAL": 0.0,   #broken-worst outcome
    "UNKNOWN": 0.5,   # unknown - give neutral score
}

def facility_summary(group):
    n = len(group)   # total number of PHCs in this state
    counts = group["status"].value_counts()   # count per status type

    def pct(key):
        # what fraction of facilities have this status?
        return counts.get(key, 0) / n if n > 0 else 0

    # Weighted average: multiply each status's count by its weight, then average.
    # A state where all PHCs are L2 gets score=1.0.
    # A state where all PHCs are NON-FUNCTIONAL gets score=0.0.
    score = sum(
        FUNC_WEIGHT.get(k, 0.5) * counts.get(k, 0)
        for k in FUNC_WEIGHT
    ) / n if n > 0 else 0

    return pd.Series({
        "total_facilities":    n,
        "pct_l2":              round(pct("L2"), 4),
        "pct_non_functional":  round(pct("NON-FUNCTIONAL"), 4),
        "functionality_score": round(score, 4)   # 0 = all broken, 1 = all perfect
    })

# groupby("state") splits the DataFrame into one group per state.
# .apply(facility_summary) runs our function on each group separately.
fac_summary = facilities.groupby("state").apply(facility_summary).reset_index()

# Pivot to wide format: 37 rows × 3 quarter columns
pivot = df.pivot_table(
    index="state",
    columns="quarter",
    values="total_alloc",
    aggfunc="sum"
).reset_index()

pivot.columns.name = None
pivot = pivot.rename(columns={
    "Q1": "total_q1",
    "Q2": "total_q2",
    "Q3": "total_q3",
})

# QUARTER-ON-QUARTER CHANGE
# How much did disbursements change from Q1→Q2 and Q2→Q3?
# Formula: (new - old) / old  — the percentage change
# e.g. Q1=₦500M, Q2=₦600M → change = (600-500)/500 = 0.20 = +20%
pivot["qoq_q2_pct"] = (
    (pivot["total_q2"] - pivot["total_q1"]) / pivot["total_q1"]
).round(4)

pivot["qoq_q3_pct"] = (
    (pivot["total_q3"] - pivot["total_q2"]) / pivot["total_q2"]
).round(4)

# Standard deviation of the 3 quarterly values for each state.
# A state whose disbursements jump around a lot has high volatility —
# which is a sign of possible misallocation or pipeline disruption.
pivot["qoq_volatility"] = pivot[["total_q1", "total_q2", "total_q3"]].std(axis=1)   #std() across columns (axis=1 = row-wise)

# GATEWAY DIVERSITY — SHANNON ENTROPY
# Shannon entropy measures how "spread out" something is.
# High entropy = funding spread evenly across all 4 gateways.
# Low entropy = one gateway dominates (e.g. 95% through NPHCDA).
# A state that depends on only one gateway is more vulnerable.
def gateway_entropy(row):
    gateway_cols = ["nphcda", "ncdc", "emtc", "nhia"]
    vals = [row[c] for c in gateway_cols]
    total = sum(vals)
    if total == 0:
        return 0.0
    probs = [v / total for v in vals]   # convert amounts to proportions (0 to 1)
    probs = [max(p, 1e-9) for p in probs]   # avoid log(0) which is undefined
    return float(entropy(probs))    # entropy() is from scipy. higher = more diverse

# Apply gateway entropy to Q3
q3_gateway = q3_clean.copy()
q3_gateway["gateway_entropy"] = q3_gateway.apply(gateway_entropy, axis=1)

# total year-to-date disbursement per state
pivot["total_ytd"] = pivot["total_q1"] + pivot["total_q2"] + pivot["total_q3"]

# Merge pivot + fac_summary + entropy into one feature table
features = pivot.merge(fac_summary, on="state", how="left")
features = features.merge(
    q3_gateway[["state", "gateway_entropy"]], on="state", how="left"
)

print("features shape:", features.shape)
features.head()

C:\Users\hp\AppData\Local\Temp\ipykernel_5800\159941024.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fac_summary = facilities.groupby("state").apply(facility_summary).reset_index()


features shape: (37, 13)


,state,total_q1,total_q2,total_q3,qoq_q2_pct,qoq_q3_pct,qoq_volatility,total_ytd,total_facilities,pct_l2,pct_non_functional,functionality_score,gateway_entropy
0,ABIA,3.055447e+08,3.055447e+08,5.803555e+08,0.0,0.8994,1.586621e+08,1.191445e+09,190.0,0.1053,0.0,0.7118,0.751553
1,ADAMAWA,4.409590e+08,4.409590e+08,8.023689e+08,0.0,0.8196,2.086601e+08,1.684287e+09,226.0,0.3186,0.0,0.8157,0.741721
2,AKWA IBOM,4.157246e+08,4.157246e+08,7.918119e+08,0.0,0.9047,2.171341e+08,1.623261e+09,316.0,0.0918,0.0,0.6522,0.706286
3,ANAMBRA,4.038091e+08,4.038091e+08,7.786860e+08,0.0,0.9284,2.164353e+08,1.586304e+09,328.0,0.0335,0.0,0.7232,0.741419
4,BAUCHI,4.885154e+08,4.885154e+08,8.830867e+08,0.0,0.8077,2.278059e+08,1.860118e+09,212.0,0.3396,0.0,0.8052,0.776328


In [36]:
# COMPOSITE ENGINEERED FEATURES
# Money per PHC: if a state has lots of facilities but low funding,
# each facility receives less — a risk factor.
features["ytd_per_facility"] = (features["total_ytd"] / features["total_facilities"])

# PARADOX FLAG: a state is "self-contradictory" if it gets HIGH funding
# but has LOW facility functionality. This shouldn't happen —
# money is supposed to improve facilities. When it doesn't, something is wrong.
median_funding = features["total_ytd"].median()   # middle value of states' funding
median_func    =    features["functionality_score"].median()   # middle functionality score

features["paradox_flag"] = (
    (features["total_ytd"] > median_funding) &         # high funding AND
    (features["functionality_score"] < median_func)    # low functionality
).astype(int)   # True -> 1, False -> 0

print("Final features shape:", features.shape)   # (37, 15)
print("Paradox states:", features[features["paradox_flag"] == 1]["state"].tolist())
features.head()


Final features shape: (37, 15)
Paradox states: ['AKWA IBOM', 'ANAMBRA', 'BENUE', 'ENUGU', 'IMO', 'KANO', 'KATSINA', 'OYO', 'RIVERS', 'SOKOTO']


,state,total_q1,total_q2,total_q3,qoq_q2_pct,qoq_q3_pct,qoq_volatility,total_ytd,total_facilities,pct_l2,pct_non_functional,functionality_score,gateway_entropy,ytd_per_facility,paradox_flag
0,ABIA,3.055447e+08,3.055447e+08,5.803555e+08,0.0,0.8994,1.586621e+08,1.191445e+09,190.0,0.1053,0.0,0.7118,0.751553,6.270762e+06,0
1,ADAMAWA,4.409590e+08,4.409590e+08,8.023689e+08,0.0,0.8196,2.086601e+08,1.684287e+09,226.0,0.3186,0.0,0.8157,0.741721,7.452597e+06,0
2,AKWA IBOM,4.157246e+08,4.157246e+08,7.918119e+08,0.0,0.9047,2.171341e+08,1.623261e+09,316.0,0.0918,0.0,0.6522,0.706286,5.136902e+06,1
3,ANAMBRA,4.038091e+08,4.038091e+08,7.786860e+08,0.0,0.9284,2.164353e+08,1.586304e+09,328.0,0.0335,0.0,0.7232,0.741419,4.836293e+06,1
4,BAUCHI,4.885154e+08,4.885154e+08,8.830867e+08,0.0,0.8077,2.278059e+08,1.860118e+09,212.0,0.3396,0.0,0.8052,0.776328,8.774139e+06,0


**The Paradox Flag is a key insight**

10 states were flagged as paradoxical — they receive above-average funding but have below-average facility health. In a well-functioning system, more money should mean better facilities. When it doesn't, the data is telling us that money may be leaving the pipeline before it reaches the facilities it was meant to support.

**POPULATION NORMALISATION & DERIVED EFFICIENCY FEATURES**
    
Adjusting for state size so funding comparisons are fair across
Nigeria's 37 states — from Kano's 13.9M people to Bayelsa's 2.3M.

In [37]:
# ── POPULATION DATA (NPC 2023 projections) ──
# Source: Nigeria National Population Commission 2023 estimates
POP_DATA = {
    "ABIA": 3727000, "ADAMAWA": 4253000, "AKWA IBOM": 5450000,
    "ANAMBRA": 5527000, "BAUCHI": 6537000, "BAYELSA": 2278000,
    "BENUE": 5742000, "BORNO": 5860000, "CROSS RIVER": 3737000,
    "DELTA": 5663000, "EBONYI": 3127000, "EDO": 4736000,
    "EKITI": 3270000, "ENUGU": 4411000, "GOMBE": 3564000,
    "IMO": 5408000, "JIGAWA": 5828000, "KADUNA": 8252000,
    "KANO": 13893000, "KATSINA": 8046000, "KEBBI": 4441000,
    "KOGI": 4473000, "KWARA": 3188000, "LAGOS": 15388000,
    "NASARAWA": 2523000, "NIGER": 6082000, "OGUN": 5765000,
    "ONDO": 4671000, "OSUN": 4714000, "OYO": 8553000,
    "PLATEAU": 4031000, "RIVERS": 7303000, "SOKOTO": 5318000,
    "TARABA": 3278000, "YOBE": 3294000, "ZAMFARA": 4349000,
    "FCT": 3564000,
}

pop_df = pd.DataFrame(POP_DATA.items(), columns=["state", "population_2023"])

# Join population into features
features = features.merge(pop_df, on="state", how="left")

# ── FEATURE: funding per million people ───
# This normalises total disbursement by population size.
# A state with ₦10B and 10M people gets the same score as
# a state with ₦5B and 5M people — fair comparison.
features["ytd_per_million_pop"] = (features["total_ytd"] / (features["population_2023"] / 1_000_000)).round(2)

# ── FEATURE: efficiency index ───
# Combines funding intensity with facility functionality.
# High score = state is getting good health outcomes per naira spent.
# Low score = state is spending a lot but facilities aren't performing.
# Formula: functionality_score / log(ytd_per_facility + 1)
# log() compresses the wide range of funding values so a state
# spending 10x more doesn't automatically score 10x higher.
import numpy as np
features["efficiency_index"] = (
    features["functionality_score"] /
    np.log1p(features["ytd_per_facility"])   # log1p = log(x + 1), safe when x=0
).round(6)

print("features shape:", features.shape) 
print(features[["state", "ytd_per_million_pop", "efficiency_index"]].head(10))


features shape: (37, 18)
         state  ytd_per_million_pop  efficiency_index
0         ABIA         3.196793e+08          0.045478
1      ADAMAWA         3.960233e+08          0.051548
2    AKWA IBOM         2.978461e+08          0.042208
3      ANAMBRA         2.870100e+08          0.046986
4       BAUCHI         2.845522e+08          0.050365
5      BAYELSA         3.742090e+08          0.046993
6        BENUE         2.771177e+08          0.047243
7        BORNO         3.507220e+08          0.048180
8  CROSS RIVER         3.398368e+08          0.051280
9        DELTA         2.340472e+08          0.050524


In [38]:
features.to_csv(OUT_DIR / "state_features.csv", index=False)
print("Saved state_features.csv —", features.shape) 

Saved state_features.csv — (37, 18)
